In [ ]:
import numpy as np
import pandas as pd
import faiss
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Kaggle Environment Setup
# Notebook ini  : Tahap 3-5  (FAISS Indexing → Retrieval → Regression)
# Prasyarat     : siamese_bilstm.ipynb sudah dijalankan dan
#                 file vectors_fold_*.npz sudah diupload sebagai dataset.
# ============================================================

VECTORS_SLUG = "bilstm-vectors"   # <-- GANTI sesuai nama dataset vectors kamu
VECTORS_DIR  = f"/kaggle/input/{VECTORS_SLUG}"
OUT_DIR      = "/kaggle/working"
N_FOLDS      = 12   # Jumlah IDPSJ / fold

print(f"TensorFlow : {tf.__version__}")
print(f"FAISS      : {faiss.__version__}")

print(f"\n=== Verifikasi File Vectors ===")
for fold_num in range(1, N_FOLDS + 1):
    path   = os.path.join(VECTORS_DIR, f'vectors_fold_{fold_num:02d}.npz')
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  vectors_fold_{fold_num:02d}.npz  -> {status}")

In [ ]:
# ============================================================
# TAHAP 3 — FAISS Indexing
# TAHAP 4 — Similarity Search (Retrieval)
# TAHAP 5 — Regression Head
# Helper functions
# ============================================================

ENC_DIM = 256   # dimensi vector dari BiLSTM encoder


def build_faiss_index(vectors: np.ndarray) -> faiss.IndexFlatL2:
    """
    TAHAP 3 — Bangun FAISS index dari vektor answerkey.
    IndexFlatL2: exact search (L2 distance), reproducible.
    Ganti dengan IndexIVFFlat untuk dataset yang lebih besar.
    """
    vectors = np.ascontiguousarray(vectors.astype(np.float32))
    index   = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    return index


def retrieve_closest(ea: np.ndarray,
                     eak_pool: np.ndarray,
                     index: faiss.IndexFlatL2,
                     k: int = 1) -> np.ndarray:
    """
    TAHAP 4 — Cari answerkey terdekat untuk setiap answer vector.
    Input : ea (N, 256) — vektor jawaban siswa
    Output: (N, 256) — vektor answerkey terdekat dari eak_pool
    """
    ea_c      = np.ascontiguousarray(ea.astype(np.float32))
    _, nn_idx = index.search(ea_c, k)   # nn_idx: (N, k)
    return eak_pool[nn_idx[:, 0]]        # ambil nearest neighbor pertama


def make_features(eq: np.ndarray, ea: np.ndarray, sj: np.ndarray) -> np.ndarray:
    """
    TAHAP 5 — Gabung fitur untuk regression head (3 input).
    Concat: [eq, ea, sj, |ea - sj|, ea ⊙ sj]  →  5 × 256D = 1280D
      eq : vektor question  (konteks pertanyaan)
      ea : vektor answer    (jawaban siswa)
      sj : vektor answerkey terdekat hasil FAISS retrieval
    """
    abs_diff = np.abs(ea - sj)
    had_prod = ea * sj
    return np.concatenate([eq, ea, sj, abs_diff, had_prod], axis=1).astype(np.float32)


def build_regression_head(enc_dim=256, dropout=0.3):
    """
    TAHAP 5 — Regression head.
    Input  : (5 × enc_dim,) = 1280D
    Output : skor regresi kontinu (1D)
    """
    inp = Input(shape=(enc_dim * 5,), name='reg_input')
    x   = Dense(256, activation='relu', name='dense_256')(inp)
    x   = Dropout(dropout,              name='dropout')(x)
    x   = Dense(64,  activation='relu', name='dense_64')(x)
    out = Dense(1, activation='linear', name='output')(x)
    model = Model(inputs=inp, outputs=out, name='regression_head')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


# Preview regression head
_tmp = build_regression_head(enc_dim=ENC_DIM)
_tmp.summary()


In [ ]:
# ============================================================
# LOPO — FAISS Retrieval + Regression (3 input: q, ak, a)
#
# Per fold:
#   Tahap 3 : Load 2D vectors → bangun FAISS index dari eak_train
#   Tahap 4 : Query FAISS → cari answerkey terdekat (sj) untuk
#             setiap answer (ea) di train/val/test
#   Tahap 5 : Fitur [eq, ea, sj, |ea-sj|, ea⊙sj] → regression → eval
# ============================================================

results = []

for fold_num in range(1, N_FOLDS + 1):
    fold_path = os.path.join(VECTORS_DIR, f'vectors_fold_{fold_num:02d}.npz')
    data      = np.load(fold_path)

    eq_train  = data['eq_train'];   eak_train = data['eak_train'];  ea_train = data['ea_train'];  y_train = data['y_train']
    eq_val    = data['eq_val'];     eak_val   = data['eak_val'];    ea_val   = data['ea_val'];    y_val   = data['y_val']
    eq_test   = data['eq_test'];    eak_test  = data['eak_test'];   ea_test  = data['ea_test'];   y_test  = data['y_test']
    test_id   = int(data['test_idpsj'][0])
    val_id    = int(data['val_idpsj'][0])

    print(f"\n{'='*60}")
    print(f"Fold {fold_num:02d}/{N_FOLDS}  |  Test={test_id}  |  Val={val_id}")
    print(f"  Data  ->  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")

    # ------------------------------------------------------------------
    # TAHAP 3 — Bangun FAISS Index dari answerkey train
    # ------------------------------------------------------------------
    faiss_index = build_faiss_index(eak_train)
    print(f"  FAISS index  ->  {faiss_index.ntotal} vektor  (dim={faiss_index.d})")

    # ------------------------------------------------------------------
    # TAHAP 4 — Retrieval: cari answerkey terdekat per answer
    # ------------------------------------------------------------------
    sj_train = retrieve_closest(ea_train, eak_train, faiss_index)
    sj_val   = retrieve_closest(ea_val,   eak_train, faiss_index)
    sj_test  = retrieve_closest(ea_test,  eak_train, faiss_index)

    # ------------------------------------------------------------------
    # TAHAP 5 — Buat fitur (inkl. eq), latih regression head, evaluasi
    # Fitur: [eq, ea, sj, |ea-sj|, ea⊙sj]  →  5 × 256D = 1280D
    # ------------------------------------------------------------------
    X_train = make_features(eq_train, ea_train, sj_train)
    X_val   = make_features(eq_val,   ea_val,   sj_val)
    X_test  = make_features(eq_test,  ea_test,  sj_test)

    reg_model = build_regression_head(enc_dim=ENC_DIM, dropout=0.3)

    es = EarlyStopping(monitor='val_loss', patience=5,
                       restore_best_weights=True, verbose=0)

    reg_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50, batch_size=32,
        callbacks=[es], verbose=1
    )

    y_pred = reg_model.predict(X_test, verbose=0).flatten()
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mae    = mean_absolute_error(y_test, y_pred)

    print(f"\n  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}")
    print(f"  Prediksi (sample 5): {np.round(y_pred[:5], 2)}")
    print(f"  Aktual   (sample 5): {y_test[:5]}")

    results.append({
        'fold': fold_num, 'test_idpsj': test_id, 'val_idpsj': val_id,
        'n_train': len(y_train), 'n_val': len(y_val), 'n_test': len(y_test),
        'rmse': rmse, 'mae': mae
    })

    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")


In [ ]:
# ============================================================
# Ringkasan Hasil LOPO (FAISS + Regression)
# ============================================================

results_df = pd.DataFrame(results)

print("=== Hasil Per-Fold (FAISS Retrieval + Regression Head) ===")
print(results_df[['fold','test_idpsj','val_idpsj','n_train','n_val','n_test',
                   'rmse','mae']].to_string(index=False))

print(f"\n{'='*60}")
print(f"Rata-rata RMSE : {results_df['rmse'].mean():.4f}  (+/-  {results_df['rmse'].std():.4f})")
print(f"Rata-rata MAE  : {results_df['mae'].mean():.4f}  (+/-  {results_df['mae'].std():.4f})")

out_path = os.path.join(OUT_DIR, 'faiss_regression_results.csv')
results_df.to_csv(out_path, index=False)
print(f"\nHasil disimpan ke {out_path}")